# 02 — Preprocessing & Unified Manifest Construction

| Field | Detail |
|---|---|
| **Notebook** | `02_preprocessing.ipynb` |
| **Pipeline stage** | H1 — Voice Emotion Recognition - 2 of 7 |
| **Owner(s)** | *Thrithwaka* |
| **Created** | *27/08/2026* |
| **Last updated** | *27/08/2026* |
| **Upstream dependency** | `01_data_acquisition.ipynb` — reads its acquisition report and raw audio directories |
| **Downstream dependency** | `03_eda.ipynb`, `04a_feature_extraction_handcrafted.ipynb`, `04b_feature_extraction_raw_audio.ipynb`, `04c_asr_transcription.ipynb` all read this notebook's output manifest |
| **Research proposal reference** | Section 5.1 (Voice Emotion Recognition — six-class scheme) |

## Purpose

This notebook is the **single source of truth** for how raw RAVDESS, TESS, and SAVEE audio becomes one unified, correctly-labeled dataset. Every downstream notebook — every feature-extraction pipeline, every one of the four candidate models, the H1 hypothesis test itself — depends on the manifest produced here being correct. Getting this notebook right once means every team member and every model works from identical, verified ground truth. Getting it wrong here would silently corrupt every result downstream, which is why label-mapping correctness is treated as the top priority of this entire notebook.

**EMO-DB is intentionally excluded from this notebook.** Per the team's agreed scope, EMO-DB is reserved as a separate, held-out cross-lingual generalization test (evaluated only after H1's primary model is selected — see `docs/research_proposal_mapping.md`), not merged into this training pool.

## Objectives

1. Load and validate the acquisition report produced by `01_data_acquisition.ipynb`.
2. Define and document, per dataset, the exact native-label-to-unified-6-class mapping, with the rationale and source for every mapping decision.
3. Parse every RAVDESS, TESS, and SAVEE filename correctly to extract: emotion label, speaker/actor ID, gender, and source dataset.
4. Validate the resulting per-class counts against the datasets' documented design (not a secondary/derived table — see the note in Section 2 on why).
5. Cross-verify file integrity against the SHA-256 checksums generated in notebook 01.
6. Decide and justify a single target sample rate and clip duration for standardization.
7. Produce a stratified train/validation/test split, fixed by a project-wide random seed.
8. Resample and duration-standardize every retained audio file into `data/processed/audio_standardized/`.
9. Write the final unified manifest (`data/processed/h1_manifest.csv`) and a machine-readable summary report.
10. Leave a structured handoff note for the next three notebooks.

## 0. Environment Setup

In [1]:
import os
import sys
import json
import re
import hashlib
import logging
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "config").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from config.settings import settings  # noqa: E402  (import after sys.path fix, matches project convention)

print(f"Project root resolved to: {PROJECT_ROOT}")

Project root resolved to: C:\Users\thrit\Desktop\emotion-ai-companion-research


In [2]:
LOG_DIR = PROJECT_ROOT / "reports" / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = LOG_DIR / "02_preprocessing.log"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[logging.FileHandler(LOG_PATH, mode="w"), logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger("preprocessing")
logger.info("Logging initialized. Log file: %s", LOG_PATH)

RANDOM_SEED = 42  # project-wide seed convention — matches 01_data_acquisition.ipynb and config/settings.py usage
np.random.seed(RANDOM_SEED)

2026-08-28 07:52:44,756 | INFO | Logging initialized. Log file: C:\Users\thrit\Desktop\emotion-ai-companion-research\reports\logs\02_preprocessing.log


## 1. Load & Validate the Acquisition Report

This notebook does not hardcode raw-data paths independently — it reads them from `reports/01_data_acquisition_report.json`, produced by the previous notebook. This is deliberate: if notebook 01 is re-run against a different directory layout, this notebook automatically follows it, rather than silently working from stale hardcoded paths.

In [3]:
acquisition_report_path = PROJECT_ROOT / "reports" / "01_data_acquisition_report.json"
if not acquisition_report_path.exists():
    raise FileNotFoundError(
        f"{acquisition_report_path} not found. Run 01_data_acquisition.ipynb to completion first."
    )

with open(acquisition_report_path) as f:
    acquisition_report = json.load(f)

REQUIRED_DATASETS = ["ravdess", "tess", "savee"]  # EMO-DB deliberately excluded — see Purpose above

RAW_DATASET_DIRS = {}
for name in REQUIRED_DATASETS:
    info = acquisition_report["datasets"][name]
    if info["verification_status"] != "OK":
        raise ValueError(
            f"Dataset '{name}' has verification_status='{info['verification_status']}' in the "
            "acquisition report. Fix and re-run 01_data_acquisition.ipynb before continuing — "
            "do not proceed with unverified raw data."
        )
    RAW_DATASET_DIRS[name] = PROJECT_ROOT / info["local_path"]
    logger.info("%s verified OK, path: %s", name, RAW_DATASET_DIRS[name])

print("All required datasets verified OK. Proceeding.")
print(RAW_DATASET_DIRS)

2026-08-28 07:52:50,900 | INFO | ravdess verified OK, path: C:\Users\thrit\Desktop\emotion-ai-companion-research\training\data\ravdess_raw
2026-08-28 07:52:50,901 | INFO | tess verified OK, path: C:\Users\thrit\Desktop\emotion-ai-companion-research\training\data\tess_raw
2026-08-28 07:52:50,902 | INFO | savee verified OK, path: C:\Users\thrit\Desktop\emotion-ai-companion-research\training\data\savee_raw
All required datasets verified OK. Proceeding.
{'ravdess': WindowsPath('C:/Users/thrit/Desktop/emotion-ai-companion-research/training/data/ravdess_raw'), 'tess': WindowsPath('C:/Users/thrit/Desktop/emotion-ai-companion-research/training/data/tess_raw'), 'savee': WindowsPath('C:/Users/thrit/Desktop/emotion-ai-companion-research/training/data/savee_raw')}


In [4]:
# Confirm the project's configured emotion label set matches exactly what this
# notebook targets — every model module in src/emotion/ reads this same list,
# so a mismatch here would silently break every downstream component.
UNIFIED_LABELS = set(settings.emotion_labels)
EXPECTED_LABELS = {"happy", "sad", "angry", "fear", "neutral", "surprise"}

assert UNIFIED_LABELS == EXPECTED_LABELS, (
    f"config/settings.py emotion_labels {UNIFIED_LABELS} does not match the "
    f"6-class scheme this notebook implements {EXPECTED_LABELS}. Reconcile before proceeding."
)
logger.info("Confirmed target label set matches config/settings.py: %s", sorted(UNIFIED_LABELS))

2026-08-28 07:52:58,693 | INFO | Confirmed target label set matches config/settings.py: ['angry', 'fear', 'happy', 'neutral', 'sad', 'surprise']


## 2. Label Mapping — Definitions, Sources, and Rationale

**This is the most important section of this notebook. Read it fully before trusting any output below it.**

RAVDESS, TESS, and SAVEE were each designed independently, by different research groups, with different native emotion vocabularies. To merge them into one 6-class scheme (happy, sad, angry, fear, neutral, surprise) we have to make — and explicitly document — two kinds of decisions per dataset: **exclusions** (native emotions with no equivalent in our scheme) and **merges** (native emotions folded into one of our 6 classes).

> **A note on sources.** The expected per-class counts used for validation in Section 4 are derived analytically from each dataset's own documented design (cited below), *not* from the secondary comparison table in Chowdhury, Ramanna & Kotecha (2025), Table 2 (*Speech emotion recognition with light weight deep neural ensemble model...*, Scientific Reports). That table's "—" (N/A) markers appear misaligned with each dataset's actually-documented emotion set when cross-checked (e.g. it implies RAVDESS has no "fear" data, which is incorrect — RAVDESS is well known to include 192 fearful samples). This is very likely a PDF-table-extraction artifact in that source, not a real property of the datasets. We flag it here rather than propagate a possibly-wrong number into our own validation.

### RAVDESS

**Source:** Livingstone, S. R., & Russo, F. A. (2018). *PLoS ONE*, 13(5).

Filenames encode 7 numeric identifiers, e.g. `03-01-06-01-02-01-12.wav`, in the order: **Modality – Vocal Channel – Emotion – Intensity – Statement – Repetition – Actor**. Since we downloaded the *speech-only* subset (per notebook 01), Modality and Vocal Channel are expected to be constant (`03`, `01`), and we verify this defensively rather than assume it.

| Code | Native emotion | → Unified label | Rationale |
|---|---|---|---|
| 01 | neutral | `neutral` | Direct match |
| 02 | calm | `neutral` | **Merge.** No "calm" class exists in our scheme; calm is acoustically and semantically closest to neutral (low arousal, non-negative valence). This exact merge is independently corroborated by Chowdhury et al. (2025), whose own reported RAVDESS neutral count of 288 only reconciles with RAVDESS's documented 96 neutral + 192 calm samples if they made the same merge — giving us external validation of this specific decision. |
| 03 | happy | `happy` | Direct match |
| 04 | sad | `sad` | Direct match |
| 05 | angry | `angry` | Direct match |
| 06 | fearful | `fear` | Direct match |
| 07 | disgust | **excluded** | No equivalent class in our 6-class scheme |
| 08 | surprised | `surprise` | Direct match |

Actor ID parity gives gender per the dataset's own documentation: odd Actor ID = male, even = female.

### TESS

**Source:** Dupuis, K., & Pichora-Fuller, M. K. (2010). Toronto Emotional Speech Set.

Both filenames and parent folder names contain the emotion as a literal word (e.g. `OAF_back_angry.wav`), with some Kaggle mirrors representing "surprise" as `ps` (pleasant surprise) rather than the word `surprise` — both are handled below.

| Native emotion | → Unified label |
|---|---|
| angry / anger | `angry` |
| disgust | **excluded** |
| fear / fearful | `fear` |
| happy / happiness | `happy` |
| neutral | `neutral` |
| sad / sadness | `sad` |
| surprise / surprised / pleasant / ps | `surprise` |

Both TESS speakers (`OAF` = older adult female, `YAF` = younger adult female) are female — TESS contains no male speakers, a known limitation worth restating in your thesis's dataset limitations section.

### SAVEE

**Source:** Jackson, P., & Haq, S. (2014). Surrey Audio-Visual Expressed Emotion Database.

Filenames use a 2-letter speaker code followed by a short emotion code and a 2-digit index, e.g. `DC_a01.wav`, `JE_sa04.wav`.

| Code | Native emotion | → Unified label |
|---|---|---|
| a | anger | `angry` |
| d | disgust | **excluded** |
| f | fear | `fear` |
| h | happiness | `happy` |
| n | neutral | `neutral` |
| sa | sadness | `sad` |
| su | surprise | `surprise` |

All 4 SAVEE speakers (`DC`, `JE`, `JK`, `KL`) are male — SAVEE contains no female speakers, the mirror-image limitation of TESS. Combining the two partially offsets each dataset's individual gender imbalance in the merged pool, which is itself worth stating explicitly as a design rationale in your methodology.

## 3. Parsing Functions

In [5]:
# --- RAVDESS -------------------------------------------------------------
RAVDESS_PATTERN = re.compile(r"^(\d{2})-(\d{2})-(\d{2})-(\d{2})-(\d{2})-(\d{2})-(\d{2})$")
RAVDESS_EMOTION_MAP = {
    "01": "neutral",
    "02": "neutral",  # calm -> neutral (merged; see Section 2 rationale)
    "03": "happy",
    "04": "sad",
    "05": "angry",
    "06": "fear",
    "07": None,        # disgust -> excluded
    "08": "surprise",
}


def parse_ravdess(filepath: Path) -> dict | None:
    stem = filepath.stem
    match = RAVDESS_PATTERN.match(stem)
    if not match:
        logger.warning("RAVDESS filename did not match expected pattern: %s", filepath.name)
        return None

    modality, vocal_channel, emotion_code, intensity, statement, repetition, actor = match.groups()
    if modality != "03" or vocal_channel != "01":
        logger.warning(
            "Unexpected modality/vocal_channel (%s/%s) in speech-only dataset: %s",
            modality, vocal_channel, filepath.name,
        )

    mapped_label = RAVDESS_EMOTION_MAP.get(emotion_code)
    actor_id = int(actor)
    gender = "male" if actor_id % 2 == 1 else "female"

    return {
        "mapped_label": mapped_label,
        "original_emotion_code": emotion_code,
        "speaker_id": f"Actor_{actor_id:02d}",
        "gender": gender,
    }

In [6]:
# --- TESS ------------------------------------------------------------------
TESS_TOKEN_MAP = {
    "angry": "angry", "anger": "angry",
    "disgust": None,
    "fear": "fear", "fearful": "fear",
    "happy": "happy", "happiness": "happy",
    "neutral": "neutral",
    "sad": "sad", "sadness": "sad",
    "surprise": "surprise", "surprised": "surprise",
    "pleasant": "surprise", "ps": "surprise",
}


def parse_tess(filepath: Path) -> dict | None:
    stem = filepath.stem.lower()
    tokens = re.split(r"[_\-]", stem)

    mapped_label = None
    matched_token = None
    for token in tokens:
        if token in TESS_TOKEN_MAP:
            mapped_label = TESS_TOKEN_MAP[token]
            matched_token = token
            break

    if matched_token is None:
        logger.warning("Could not identify emotion token in TESS filename: %s", filepath.name)
        return None

    speaker_code = tokens[0].upper() if tokens else "UNKNOWN"
    speaker_map = {"OAF": "OAF (older adult female)", "YAF": "YAF (younger adult female)"}

    return {
        "mapped_label": mapped_label,
        "original_emotion_code": matched_token,
        "speaker_id": speaker_map.get(speaker_code, speaker_code),
        "gender": "female",  # both TESS speakers are female — see Section 2
    }

In [7]:
# --- SAVEE -------------------------------------------------------------------
SAVEE_PATTERN = re.compile(r"^([A-Za-z]{2})_([A-Za-z]+)(\d+)$")
SAVEE_EMOTION_MAP = {
    "a": "angry",
    "d": None,       # disgust -> excluded
    "f": "fear",
    "h": "happy",
    "n": "neutral",
    "sa": "sad",
    "su": "surprise",
}


def parse_savee(filepath: Path) -> dict | None:
    stem = filepath.stem
    match = SAVEE_PATTERN.match(stem)
    if not match:
        logger.warning("SAVEE filename did not match expected pattern: %s", filepath.name)
        return None

    speaker_code, emotion_code, _index = match.groups()
    emotion_code = emotion_code.lower()
    mapped_label = SAVEE_EMOTION_MAP.get(emotion_code)

    if mapped_label is None and emotion_code not in SAVEE_EMOTION_MAP:
        logger.warning("Unrecognized SAVEE emotion code '%s' in file: %s", emotion_code, filepath.name)

    return {
        "mapped_label": mapped_label,
        "original_emotion_code": emotion_code,
        "speaker_id": speaker_code.upper(),
        "gender": "male",  # all 4 SAVEE speakers are male — see Section 2
    }


PARSERS = {"ravdess": parse_ravdess, "tess": parse_tess, "savee": parse_savee}

## 4. Build the Unified Manifest

Every `.wav` file in each raw dataset directory is parsed. Files whose emotion maps to `None` (i.e. excluded native emotions like disgust) are kept in a separate `excluded_df` for transparent reporting, rather than silently dropped — this is what lets us defend the exact exclusion counts in a viva.

In [8]:
AUDIO_EXTENSIONS = {".wav"}

included_rows = []
excluded_rows = []
unparsed_rows = []

for source_dataset, raw_dir in RAW_DATASET_DIRS.items():
    parser = PARSERS[source_dataset]
    audio_files = sorted(f for f in raw_dir.rglob("*") if f.suffix.lower() in AUDIO_EXTENSIONS)
    logger.info("Parsing %d files from %s...", len(audio_files), source_dataset)

    for filepath in audio_files:
        parsed = parser(filepath)
        relative_path = str(filepath.relative_to(PROJECT_ROOT))

        if parsed is None:
            unparsed_rows.append({"source_dataset": source_dataset, "original_path": relative_path})
            continue

        row = {
            "original_path": relative_path,
            "source_dataset": source_dataset,
            "speaker_id": parsed["speaker_id"],
            "gender": parsed["gender"],
            "original_emotion_code": parsed["original_emotion_code"],
            "label": parsed["mapped_label"],
        }
        if parsed["mapped_label"] is None:
            excluded_rows.append(row)
        else:
            included_rows.append(row)

included_df = pd.DataFrame(included_rows)
excluded_df = pd.DataFrame(excluded_rows)
unparsed_df = pd.DataFrame(unparsed_rows)

print(f"Included (retained for training): {len(included_df)} files")
print(f"Excluded (native emotion has no equivalent class): {len(excluded_df)} files")
print(f"Unparsed (filename did not match expected pattern — investigate!): {len(unparsed_df)} files")

if len(unparsed_df) > 0:
    logger.error("%d files could not be parsed — see log for filenames.", len(unparsed_df))
    display(unparsed_df)

2026-08-28 07:53:21,236 | INFO | Parsing 1440 files from ravdess...
2026-08-28 07:53:21,380 | INFO | Parsing 2800 files from tess...
2026-08-28 07:53:21,440 | INFO | Parsing 480 files from savee...
Included (retained for training): 4068 files
Excluded (native emotion has no equivalent class): 652 files
Unparsed (filename did not match expected pattern — investigate!): 0 files


> **If `unparsed_df` is non-empty:** do not proceed. This means a filename didn't match the expected pattern for its dataset — usually a sign the Kaggle mirror's naming convention differs slightly from what's documented above, or a non-audio/README file got picked up by the glob. Inspect the listed files directly and adjust the relevant `parse_*` function's regex before continuing.

In [9]:
print("Excluded files by dataset and native emotion code (should be exactly the disgust class in each dataset):")
excluded_df.groupby(["source_dataset", "original_emotion_code"]).size().rename("count").reset_index()

Excluded files by dataset and native emotion code (should be exactly the disgust class in each dataset):


,source_dataset,original_emotion_code,count
0,ravdess,07,192
1,savee,d,60
2,tess,disgust,400


## 5. Validate Class Counts Against Documented Dataset Design

These expected counts are derived analytically in Section 2 from each dataset's own published design, independent of the potentially-misaligned secondary table discussed there. A mismatch here means either the label-mapping logic has a bug, or the raw data itself is incomplete/corrupted (in which case, re-check notebook 01's verification output first).

In [10]:
EXPECTED_COUNTS_AFTER_MAPPING = {
    "ravdess": {"neutral": 288, "happy": 192, "sad": 192, "angry": 192, "fear": 192, "surprise": 192},
    "tess":    {"neutral": 400, "happy": 400, "sad": 400, "angry": 400, "fear": 400, "surprise": 400},
    "savee":   {"neutral": 120, "happy": 60,  "sad": 60,  "angry": 60,  "fear": 60,  "surprise": 60},
}

actual_counts = (
    included_df.groupby(["source_dataset", "label"]).size().unstack(fill_value=0)
)
display(actual_counts)

all_match = True
for dataset, expected_labels in EXPECTED_COUNTS_AFTER_MAPPING.items():
    for label, expected_count in expected_labels.items():
        actual_count = int(actual_counts.loc[dataset, label]) if (dataset in actual_counts.index and label in actual_counts.columns) else 0
        if actual_count != expected_count:
            all_match = False
            logger.warning(
                "MISMATCH: %s / %s — expected %d, got %d",
                dataset, label, expected_count, actual_count,
            )

if all_match:
    logger.info("All per-dataset, per-class counts match documented expectations exactly.")
    print("\u2705 All class counts validated successfully against documented dataset design.")
else:
    print("\u26a0\ufe0f  One or more class counts did not match expectations — see warnings above and the log file before proceeding.")

total_expected = sum(sum(d.values()) for d in EXPECTED_COUNTS_AFTER_MAPPING.values())
print(f"\nTotal expected after mapping + exclusion: {total_expected}")
print(f"Total actual in included_df:               {len(included_df)}")

label,angry,fear,happy,neutral,sad,surprise
source_dataset,,,,,,
ravdess,192,192,192,288,192,192
savee,60,60,60,120,60,60
tess,400,400,400,400,400,400


2026-08-28 07:53:22,400 | INFO | All per-dataset, per-class counts match documented expectations exactly.
✅ All class counts validated successfully against documented dataset design.

Total expected after mapping + exclusion: 4068
Total actual in included_df:               4068


## 6. Checksum Cross-Verification

Recomputes SHA-256 for every retained file and compares it against the manifest generated in `01_data_acquisition.ipynb`. This guards against silent file corruption or accidental modification between acquisition and preprocessing — a low-cost check given how easily this could otherwise go unnoticed.

In [11]:
checksums_path = PROJECT_ROOT / "training" / "data" / "checksums.json"
with open(checksums_path) as f:
    checksums = json.load(f)


def sha256_of_file(path: Path, chunk_size: int = 8192) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()


mismatches = []
for _, row in included_df.iterrows():
    dataset = row["source_dataset"]
    rel_path = row["original_path"]
    expected_hash = checksums.get(dataset, {}).get(rel_path)
    if expected_hash is None:
        mismatches.append((rel_path, "not found in checksum manifest"))
        continue
    actual_hash = sha256_of_file(PROJECT_ROOT / rel_path)
    if actual_hash != expected_hash:
        mismatches.append((rel_path, "hash mismatch"))

if mismatches:
    logger.error("%d checksum mismatches found:", len(mismatches))
    for path, reason in mismatches[:20]:
        logger.error("  %s: %s", path, reason)
    print(f"\u26a0\ufe0f  {len(mismatches)} checksum mismatches — see log. Do not proceed until resolved.")
else:
    logger.info("All %d retained files match their notebook-01 checksums.", len(included_df))
    print(f"\u2705 All {len(included_df)} retained files verified against notebook 01's checksum manifest.")

2026-08-28 07:54:00,726 | INFO | All 4068 retained files match their notebook-01 checksums.
✅ All 4068 retained files verified against notebook 01's checksum manifest.


## 7. Audio Duration Profiling

A quick pass using `soundfile.info()` (metadata only — does not load full audio into memory) to inform the target-duration decision below. Full visual distributions (histograms) are deferred to `03_eda.ipynb`; this section only needs enough information to justify a number.

In [12]:
def get_duration_seconds(path: Path) -> float:
    info = sf.info(str(path))
    return info.frames / info.samplerate


duration_rows = []
for _, row in included_df.iterrows():
    duration_rows.append({
        "source_dataset": row["source_dataset"],
        "duration_sec": get_duration_seconds(PROJECT_ROOT / row["original_path"]),
    })

duration_df = pd.DataFrame(duration_rows)
duration_summary = duration_df.groupby("source_dataset")["duration_sec"].describe()[["min", "mean", "50%", "max"]]
duration_summary.columns = ["min_sec", "mean_sec", "median_sec", "max_sec"]
display(duration_summary)

,min_sec,mean_sec,median_sec,max_sec
source_dataset,,,,
ravdess,2.936271,3.663569,3.636958,5.105104
savee,1.630907,3.822978,3.647800,7.138730
tess,1.254076,1.989556,1.991808,2.984804


## 8. Standardization Parameters — Decision & Rationale

Based on the duration profile above (all three datasets fall in the 2–4 second range, consistent with their own documentation) and the target of feeding identical time-domain audio into every H1 candidate model:

- **Target sample rate: 16,000 Hz.** Chosen specifically because Wav2Vec2 (one of the four H1 candidate models) requires 16 kHz input natively — using this rate for *all* models avoids maintaining two separately-resampled copies of the dataset, and Librosa-based hand-crafted feature extraction (MFCC/RMSE/ZCR/Chroma) works correctly at this rate.
- **Target duration: 2.5 seconds.** Matches the clip length used by Chowdhury et al. (2025) on these same datasets, and comfortably covers the median duration observed above without excessive padding.
- **Offset: 0.6 seconds.** Skips a short leading segment before the fixed-duration window is taken, reducing the chance of capturing leading silence — following the same practice as Chowdhury et al. (2025). If skipping 0.6s leaves less than the target duration remaining, the window falls back to starting at 0.

These constants are defined once here and are the same values every downstream feature-extraction notebook (`04a`, `04b`) will assume — do not redefine them independently elsewhere.

In [13]:
TARGET_SAMPLE_RATE = 16000
TARGET_DURATION_SECONDS = 2.5
OFFSET_SECONDS = 0.6
TARGET_LENGTH_SAMPLES = int(TARGET_SAMPLE_RATE * TARGET_DURATION_SECONDS)

STANDARDIZED_AUDIO_DIR = PROJECT_ROOT / "data" / "processed" / "audio_standardized"
STANDARDIZED_AUDIO_DIR.mkdir(parents=True, exist_ok=True)

logger.info(
    "Standardization params: sr=%d, duration=%.1fs, offset=%.1fs, target_length_samples=%d",
    TARGET_SAMPLE_RATE, TARGET_DURATION_SECONDS, OFFSET_SECONDS, TARGET_LENGTH_SAMPLES,
)

2026-08-28 07:54:02,068 | INFO | Standardization params: sr=16000, duration=2.5s, offset=0.6s, target_length_samples=40000


## 9. Stratified Train / Validation / Test Split

An 80/10/10 split, stratified on a combined `label + source_dataset` key so that both the emotion-class balance *and* the dataset-origin balance are preserved across all three splits — not just the label balance alone. This matters here specifically because SAVEE is entirely male and TESS is entirely female; without stratifying on source too, a naive label-only split could leave the test set skewed toward one gender, which is exactly the pitfall Paper 1's review discussion warns is common in single-corpus SER evaluation.

In [14]:
included_df = included_df.reset_index(drop=True)
stratify_key = included_df["label"] + "|" + included_df["source_dataset"]

train_df, temp_df = train_test_split(
    included_df, test_size=0.20, stratify=stratify_key, random_state=RANDOM_SEED
)
temp_stratify_key = temp_df["label"] + "|" + temp_df["source_dataset"]
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_stratify_key, random_state=RANDOM_SEED
)

train_df = train_df.copy(); train_df["split"] = "train"
val_df = val_df.copy(); val_df["split"] = "val"
test_df = test_df.copy(); test_df["split"] = "test"

manifest_df = pd.concat([train_df, val_df, test_df], ignore_index=True)

print("Split sizes:")
print(manifest_df["split"].value_counts())
print("\nLabel distribution per split (should be proportionally similar across rows):")
display(pd.crosstab(manifest_df["split"], manifest_df["label"]))
print("\nSource-dataset distribution per split (should be proportionally similar across rows):")
display(pd.crosstab(manifest_df["split"], manifest_df["source_dataset"]))

Split sizes:
split
train    3254
val       407
test      407
Name: count, dtype: int64

Label distribution per split (should be proportionally similar across rows):


label,angry,fear,happy,neutral,sad,surprise
split,,,,,,
test,65,65,66,81,65,65
train,522,522,521,646,522,521
val,65,65,65,81,65,66



Source-dataset distribution per split (should be proportionally similar across rows):


source_dataset,ravdess,savee,tess
split,,,
test,125,42,240
train,998,336,1920
val,125,42,240


## 10. Audio Standardization — Resample, Trim/Pad, Save

Every retained file is loaded, resampled to 16 kHz, trimmed or padded to exactly 2.5 seconds using the offset defined above, and written to `data/processed/audio_standardized/<source_dataset>/`. This is the **shared audio artifact** every H1 model notebook will load from — feature extraction notebooks should never read from `training/data/*_raw/` directly.

In [15]:
def standardize_audio(input_path: Path, output_path: Path) -> None:
    y, _ = librosa.load(str(input_path), sr=TARGET_SAMPLE_RATE)

    offset_samples = int(OFFSET_SECONDS * TARGET_SAMPLE_RATE)
    if len(y) > offset_samples + TARGET_LENGTH_SAMPLES:
        y = y[offset_samples : offset_samples + TARGET_LENGTH_SAMPLES]
    elif len(y) > TARGET_LENGTH_SAMPLES:
        y = y[:TARGET_LENGTH_SAMPLES]

    if len(y) < TARGET_LENGTH_SAMPLES:
        y = np.pad(y, (0, TARGET_LENGTH_SAMPLES - len(y)), mode="constant")

    output_path.parent.mkdir(parents=True, exist_ok=True)
    sf.write(str(output_path), y, TARGET_SAMPLE_RATE)


standardized_paths = []
total = len(manifest_df)
for i, (_, row) in enumerate(manifest_df.iterrows()):
    input_path = PROJECT_ROOT / row["original_path"]
    output_filename = Path(row["original_path"]).stem + ".wav"
    output_path = STANDARDIZED_AUDIO_DIR / row["source_dataset"] / output_filename

    standardize_audio(input_path, output_path)
    standardized_paths.append(str(output_path.relative_to(PROJECT_ROOT)))

    if (i + 1) % 500 == 0 or (i + 1) == total:
        logger.info("Standardized %d / %d files", i + 1, total)

manifest_df["standardized_path"] = standardized_paths
manifest_df["duration_sec_original"] = manifest_df["original_path"].apply(
    lambda p: get_duration_seconds(PROJECT_ROOT / p)
)

print(f"Standardization complete. {total} files written to {STANDARDIZED_AUDIO_DIR}")

2026-08-28 07:54:10,123 | INFO | Standardized 500 / 4068 files
2026-08-28 07:54:13,461 | INFO | Standardized 1000 / 4068 files
2026-08-28 07:54:18,246 | INFO | Standardized 1500 / 4068 files
2026-08-28 07:54:22,601 | INFO | Standardized 2000 / 4068 files
2026-08-28 07:54:27,102 | INFO | Standardized 2500 / 4068 files
2026-08-28 07:54:31,609 | INFO | Standardized 3000 / 4068 files
2026-08-28 07:54:34,856 | INFO | Standardized 3500 / 4068 files
2026-08-28 07:54:39,101 | INFO | Standardized 4000 / 4068 files
2026-08-28 07:54:39,563 | INFO | Standardized 4068 / 4068 files
Standardization complete. 4068 files written to C:\Users\thrit\Desktop\emotion-ai-companion-research\data\processed\audio_standardized


## 11. Final Sanity Checks

Confirms every row in the manifest has a corresponding standardized file on disk, and that no unexpected labels leaked through.

In [16]:
missing_standardized = [
    p for p in manifest_df["standardized_path"] if not (PROJECT_ROOT / p).exists()
]
assert not missing_standardized, f"{len(missing_standardized)} standardized files are missing on disk!"

unexpected_labels = set(manifest_df["label"].unique()) - EXPECTED_LABELS
assert not unexpected_labels, f"Unexpected labels found in manifest: {unexpected_labels}"

print("\u2705 Every manifest row has a corresponding standardized audio file on disk.")
print("\u2705 All manifest labels are within the expected 6-class scheme.")
print(f"\nFinal manifest shape: {manifest_df.shape}")
manifest_df.head()

✅ Every manifest row has a corresponding standardized audio file on disk.
✅ All manifest labels are within the expected 6-class scheme.

Final manifest shape: (4068, 9)


,original_path,source_dataset,speaker_id,gender,original_emotion_code,label,split,standardized_path,duration_sec_original
0,training\data\ravdess_raw\Actor_13\03-01-02-01...,ravdess,Actor_13,male,02,neutral,train,data\processed\audio_standardized\ravdess\03-0...,2.936271
1,training\data\tess_raw\TESS Toronto emotional ...,tess,YAF (younger adult female),female,happy,happy,train,data\processed\audio_standardized\tess\YAF_dat...,1.823011
2,training\data\tess_raw\TESS Toronto emotional ...,tess,OAF (older adult female),female,sad,sad,train,data\processed\audio_standardized\tess\OAF_rag...,2.853895
3,training\data\tess_raw\TESS Toronto emotional ...,tess,YAF (younger adult female),female,happy,happy,train,data\processed\audio_standardized\tess\YAF_hus...,1.971738
4,training\data\tess_raw\TESS Toronto emotional ...,tess,OAF (older adult female),female,sad,sad,train,data\processed\audio_standardized\tess\OAF_lat...,2.390186


## 12. Save the Manifest and Summary Report

In [17]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

manifest_columns = [
    "original_path", "standardized_path", "source_dataset", "speaker_id", "gender",
    "original_emotion_code", "label", "duration_sec_original", "split",
]
manifest_path = PROCESSED_DIR / "h1_manifest.csv"
manifest_df[manifest_columns].to_csv(manifest_path, index=False)
logger.info("Manifest written to %s (%d rows)", manifest_path, len(manifest_df))

preprocessing_report = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "random_seed": RANDOM_SEED,
    "source_datasets": REQUIRED_DATASETS,
    "excluded_native_emotion": "disgust (all three datasets — no equivalent class in the unified scheme)",
    "ravdess_merge_note": "native 'calm' merged into 'neutral'",
    "total_included_files": int(len(manifest_df)),
    "total_excluded_files": int(len(excluded_df)),
    "total_unparsed_files": int(len(unparsed_df)),
    "class_count_validation_passed": bool(all_match),
    "checksum_verification_passed": bool(len(mismatches) == 0),
    "standardization": {
        "sample_rate_hz": TARGET_SAMPLE_RATE,
        "duration_seconds": TARGET_DURATION_SECONDS,
        "offset_seconds": OFFSET_SECONDS,
    },
    "split_sizes": manifest_df["split"].value_counts().to_dict(),
    "manifest_path": str(manifest_path.relative_to(PROJECT_ROOT)),
    "standardized_audio_dir": str(STANDARDIZED_AUDIO_DIR.relative_to(PROJECT_ROOT)),
}

report_path = PROJECT_ROOT / "reports" / "02_preprocessing_report.json"
with open(report_path, "w") as f:
    json.dump(preprocessing_report, f, indent=2)

logger.info("Preprocessing report written to %s", report_path)
print(f"Manifest: {manifest_path}")
print(f"Report:   {report_path}")
print(json.dumps(preprocessing_report, indent=2))

2026-08-28 07:54:41,493 | INFO | Manifest written to C:\Users\thrit\Desktop\emotion-ai-companion-research\data\processed\h1_manifest.csv (4068 rows)
2026-08-28 07:54:41,497 | INFO | Preprocessing report written to C:\Users\thrit\Desktop\emotion-ai-companion-research\reports\02_preprocessing_report.json
Manifest: C:\Users\thrit\Desktop\emotion-ai-companion-research\data\processed\h1_manifest.csv
Report:   C:\Users\thrit\Desktop\emotion-ai-companion-research\reports\02_preprocessing_report.json
{
  "generated_at_utc": "2026-08-28T02:24:41.494746+00:00",
  "random_seed": 42,
  "source_datasets": [
    "ravdess",
    "tess",
    "savee"
  ],
  "excluded_native_emotion": "disgust (all three datasets \u2014 no equivalent class in the unified scheme)",
  "ravdess_merge_note": "native 'calm' merged into 'neutral'",
  "total_included_files": 4068,
  "total_excluded_files": 652,
  "total_unparsed_files": 0,
  "class_count_validation_passed": true,
  "checksum_verification_passed": true,
  "stand

---

## Summary Note — Handoff to Next Notebooks

### What was done in this notebook

1. Loaded and validated the acquisition report from `01_data_acquisition.ipynb`; confirmed RAVDESS, TESS, and SAVEE all passed verification before proceeding.
2. Defined and fully documented the native-label-to-unified-6-class mapping for each dataset, citing each dataset's original paper as the source of truth (Section 2).
3. Parsed every filename in all three datasets, separating cleanly-mapped files (`included_df`), files whose native emotion has no equivalent class (`excluded_df` — disgust, in all three datasets), and any files that failed to parse (`unparsed_df` — should be empty; if not, this run did not proceed further and needs manual investigation first).
4. Validated the resulting per-class, per-dataset counts against each dataset's analytically-derived, documented design — see Section 4/5 output for this run's pass/fail result.
5. Cross-verified every retained file's SHA-256 hash against notebook 01's checksum manifest — see Section 6 output for this run's result.
6. Profiled audio durations and used them to justify the standardization parameters: 16,000 Hz sample rate, 2.5-second duration, 0.6-second offset.
7. Produced an 80/10/10 train/val/test split, stratified jointly on emotion label and source dataset, using a fixed random seed (42).
8. Resampled, trimmed/padded, and saved every retained file to `data/processed/audio_standardized/<dataset>/`.
9. Wrote the final unified manifest and a machine-readable summary report.

### Outputs produced by this notebook (and where to find them)

| Output | Location | Used by |
|---|---|---|
| Unified manifest | `data/processed/h1_manifest.csv` | `03_eda.ipynb`, `04a`, `04b`, `04c`, and every model-training notebook |
| Standardized audio (16kHz, 2.5s, all 3 datasets) | `data/processed/audio_standardized/{ravdess,tess,savee}/` | `04a_feature_extraction_handcrafted.ipynb`, `04b_feature_extraction_raw_audio.ipynb`, `04c_asr_transcription.ipynb` |
| Preprocessing report | `reports/02_preprocessing_report.json` | Sanity-checked at the start of downstream notebooks; cite directly in your thesis methodology section for exact split sizes and validation status |
| Run log | `reports/logs/02_preprocessing.log` | Debugging any downstream "file not found" or unexpected-label errors |

### Manifest schema (for reference by every downstream notebook)

| Column | Description |
|---|---|
| `original_path` | Path to the raw, un-standardized source file (for provenance only — do not load audio from here downstream) |
| `standardized_path` | Path to the 16kHz, 2.5s-standardized `.wav` — **this is what every feature-extraction notebook should read** |
| `source_dataset` | `ravdess` / `tess` / `savee` |
| `speaker_id` | Actor/speaker identifier within the source dataset |
| `gender` | `male` / `female`, per each dataset's documented speaker roster |
| `original_emotion_code` | The native label/code before mapping (for auditability) |
| `label` | The unified 6-class label — this is the training target |
| `duration_sec_original` | Duration of the original (pre-standardization) clip |
| `split` | `train` / `val` / `test` — fixed by this notebook, must not be re-randomized downstream |

### What needs to be done next

Proceed to **`03_eda.ipynb`**, which should:

- Load `data/processed/h1_manifest.csv` directly — no re-parsing of raw data.
- Visualize class distribution (overall and per split, per source dataset) as histograms/bar charts.
- Visualize duration distributions (using `duration_sec_original`) as histograms.
- Visualize a handful of sample waveforms/spectrograms per emotion class, for the thesis's data-exploration section.
- Explicitly report and visualize any residual class imbalance (e.g. RAVDESS's neutral class is larger than its other classes at 288 vs 192, post-merge) so the team can decide whether class weighting is needed before training in `05a`–`05e`.

**Before running `03_eda.ipynb`, confirm:**
- Section 4's `unparsed_df` was empty for this run.
- Section 5 printed "All per-dataset, per-class counts match documented expectations exactly."
- Section 6 printed "All ... files verified against notebook 01's checksum manifest."
- `reports/02_preprocessing_report.json` exists and its `class_count_validation_passed` and `checksum_verification_passed` fields both read `true`.

### Resources needed for the next step

| Resource | Needed for | Notes |
|---|---|---|
| `data/processed/h1_manifest.csv` | The single input every downstream H1 notebook needs | Do not regenerate independently — always read this file |
| `data/processed/audio_standardized/` | Source audio for all feature extraction | Already at the target sample rate/duration — no further resampling needed downstream |
| `matplotlib`, `seaborn` (already in `requirements.txt`) | Visualizations in `03_eda.ipynb` | No new installs required |
| `config/settings.py` (`emotion_labels`) | Confirms label consistency in any downstream notebook that re-checks it | Already validated against this notebook's scheme in Section 1 |

### Known issues / things to watch for

- *Non*

### Run metadata

- **Run by:** *Thrithwaka*
- **Date:** *27/08/2026*
- **Class count validation passed:** *Yes*
- **Checksum verification passed:** *Yes*
- **Final manifest size:** *4068*